# ETF OOS Pipeline - BASE_DATE 동적 분리 + Target Threshold Optuna

이 노트북은 기존 흐름을 유지하되, 데이터 누수를 줄이도록 기간과 Optuna 구조를 정리한 버전입니다.

흐름:

1. `BASE_DATE` 기준으로 Train / Optuna Validation / Simulation Test 자동 분리  
2. Base feature dataset 생성  
3. Train 구간 기준 VIF 제거  
4. Optuna trial 안에서 `target_return_threshold` 선택  
5. 선택된 threshold 기준으로 target 생성  
6. Train 최근 N년 기준 best lag / top feature 선택  
7. Train 전체로 학습, Optuna Validation으로 best config 선택  
8. best config 그대로 Simulation Test 평가  
9. best config로 전체 과거 재학습 후 Real Inference  

중요:  
`target_return_threshold`는 더 이상 고정 설정값이 아닙니다. Optuna가 고릅니다.

## 이번 버전의 핵심 수정

- `pred_proba_threshold`는 Optuna 안에 남깁니다.
- 대신 Optuna score를 `precision`이 아니라 `precision_lift = precision - positive_rate`로 계산합니다.
- 전부 1 예측은 `precision == positive_rate`가 되므로 score가 0이 됩니다.
- 시뮬레이션은 `n_days` 뒤 매도 로직을 적용합니다.
- actual target 기준 if 시뮬레이션은 `cash_if`, `cum_return_if` 등 `_if` 컬럼으로 같이 계산합니다.
- Plotly 그래프는 `pred vs actual_target`, `cash vs cash_if` 두 개를 생성합니다.

## Objective 수정

이번 버전의 Optuna score는 아래 방식입니다.

```python
positive_rate = actual_1_count / eval_count
precision_lift = precision - positive_rate
score = max(0, precision_lift) * recall
```

의미:

- 전부 1 예측은 `precision == positive_rate`라서 score가 0입니다.
- 완벽 예측은 `precision=1`, `recall=1`이라서 높은 score를 받습니다.
- 신호를 너무 적게 내는 모델은 precision이 높아도 recall이 낮아서 score가 낮아집니다.
- `pred_proba_threshold`는 Optuna 안에 그대로 남아 있습니다.

In [25]:
import warnings
warnings.filterwarnings("ignore")

import os
import json
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import FinanceDataReader as fdr
import optuna
import plotly.graph_objects as go

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    log_loss,
    confusion_matrix,
)
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

def display_df(df, n=20):
    try:
        display(df.head(n))
    except NameError:
        print(df.head(n))

## 0. 기존 공통 함수

파일 분리는 나중에 하기로 했으므로, 기존 `lib.py`의 함수들을 노트북 안에 포함합니다.

In [26]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import FinanceDataReader as fdr

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    log_loss,
    confusion_matrix
)
from sklearn.inspection import permutation_importance
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler

## 함수

################################################################################################################################################
## 1. Base Feature Dataset 생성
################################################################################################################################################
def load_price_data(ticker, start_date="2020-01-01", end_date=None):
    """
    FinanceDataReader로 가격 데이터를 가져온다.
    Date 컬럼을 일반 컬럼으로 유지한다.
    """
    df = fdr.DataReader(ticker, start_date, end_date)
    df = df.reset_index().rename(columns={"index": "Date"})

    # 컬럼명 정리
    df["Date"] = pd.to_datetime(df["Date"])

    return df

def make_target_etf_features(etf_df, prefix):
    """
    예측 대상 ETF용 feature 생성.
    target은 여기서 만들지 않는다.
    """
    df = etf_df.copy()
    df = df.sort_values("Date").reset_index(drop=True)

    close = df["Adj Close"]
    volume = df["Volume"]

    result = pd.DataFrame()
    result["Date"] = df["Date"]

    # target 만들 때 필요하므로 Close는 반드시 보존
    result[f"{prefix}_adj_close"] = close

    # 수익률
    result[f"{prefix}_ret_1d"] = close.pct_change(1)
    result[f"{prefix}_ret_5d"] = close.pct_change(5)
    result[f"{prefix}_ret_20d"] = close.pct_change(20)

    # 이동평균 대비 위치
    ma_5 = close.rolling(5).mean()
    ma_20 = close.rolling(20).mean()
    ma_60 = close.rolling(60).mean()

    result[f"{prefix}_ma5_ratio"] = close / ma_5 - 1
    result[f"{prefix}_ma20_ratio"] = close / ma_20 - 1
    result[f"{prefix}_ma60_ratio"] = close / ma_60 - 1

    # 변동성
    result[f"{prefix}_vol_20d"] = result[f"{prefix}_ret_1d"].rolling(20).std()

    # 거래량 비율
    vol_ma20 = volume.rolling(20).mean()
    result[f"{prefix}_volume_ratio_20d"] = volume / vol_ma20 - 1

    return result

def make_external_features(raw_df, name, feature_type="price"):
    """
    외부 지표용 최소 파생변수 생성.
    feature_type:
        - price: 일반 가격형 지표
        - risk: VIX 같은 리스크 레벨 지표
        - rate: 금리 지표
    """
    df = raw_df.copy()
    df = df.sort_values("Date").reset_index(drop=True)

    close = df["Adj Close"]

    result = pd.DataFrame()
    result["Date"] = df["Date"]

    if feature_type == "price":
        result[f"{name}_ret_5d"] = close.pct_change(5)
        result[f"{name}_ret_20d"] = close.pct_change(20)

    elif feature_type == "risk":
        result[f"{name}_level"] = close
        result[f"{name}_chg_5d"] = close.diff(5)
        result[f"{name}_chg_20d"] = close.diff(20)

    elif feature_type == "rate":
        result[f"{name}_level"] = close
        result[f"{name}_diff_5d"] = close.diff(5)
        result[f"{name}_diff_20d"] = close.diff(20)

    else:
        raise ValueError("feature_type must be one of ['price', 'risk', 'rate']")

    return result

def make_base_feature_dataset(
    etf_code,
    external_tickers,
    external_feature_types,
    start_date="2020-01-01",
    end_date=None
):
    """
    target 없는 기본 feature dataset 생성.
    이 함수는 느린 작업이므로 한 번만 실행하는 것을 목표로 한다.
    """
    # 1. ETF 본체 로드
    etf_raw = load_price_data(etf_code, start_date, end_date)

    # 2. ETF 본체 feature 생성
    base_df = make_target_etf_features(etf_raw, prefix=etf_code)

    # 3. 외부 지표 붙이기
    for name, ticker in external_tickers.items():
        print(f"Loading external ticker: {name} / {ticker}")

        try:
            raw = load_price_data(ticker, start_date, end_date)

            feature_type = external_feature_types.get(name, "price")

            ext_feat = make_external_features(
                raw_df=raw,
                name=name,
                feature_type=feature_type
            )

            base_df = base_df.merge(ext_feat, on="Date", how="left")

        except Exception as e:
            print(f"[SKIP] {name} / {ticker} 로드 실패:", e)

    # 4. 날짜 정렬
    base_df = base_df.sort_values("Date").reset_index(drop=True)

    # 5. feature_cols 정리
    close_col = f"{etf_code}_adj_close"

    feature_cols = [
        col for col in base_df.columns
        if col not in ["Date", close_col]
    ]

    return base_df, feature_cols, close_col

def make_base_feature_dataset(
    etf_code,
    external_tickers,
    external_feature_types,
    start_date="2020-01-01",
    end_date=None
):
    """
    target 없는 기본 feature dataset 생성.
    이 함수는 느린 작업이므로 한 번만 실행하는 것을 목표로 한다.

    주말 데이터가 섞여 NA가 늘어나는 문제를 막기 위해
    ETF / 외부 ticker 모두 영업일(월~금)만 사용한다.
    """

    # =========================================================
    # 0. 영업일 필터 함수
    # =========================================================
    def keep_weekdays_only(df, date_col="Date"):
        df = df.copy()
        df[date_col] = pd.to_datetime(df[date_col])
        df = df[df[date_col].dt.weekday < 5]  # 월=0, 금=4
        df = df.sort_values(date_col).reset_index(drop=True)
        return df

    # =========================================================
    # 1. ETF 본체 로드
    # =========================================================
    etf_raw = load_price_data(etf_code, start_date, end_date)

    # 주말 제거
    etf_raw = keep_weekdays_only(etf_raw, date_col="Date")

    # =========================================================
    # 2. ETF 본체 feature 생성
    # =========================================================
    base_df = make_target_etf_features(etf_raw, prefix=etf_code)

    # feature 생성 후에도 혹시 모르니 다시 주말 제거
    base_df = keep_weekdays_only(base_df, date_col="Date")

    # =========================================================
    # 3. 외부 지표 붙이기
    # =========================================================
    for name, ticker in external_tickers.items():
        print(f"Loading external ticker: {name} / {ticker}")

        try:
            raw = load_price_data(ticker, start_date, end_date)

            # 외부 ticker도 주말 제거
            raw = keep_weekdays_only(raw, date_col="Date")

            feature_type = external_feature_types.get(name, "price")

            ext_feat = make_external_features(
                raw_df=raw,
                name=name,
                feature_type=feature_type
            )

            # 외부 feature 생성 후에도 다시 주말 제거
            ext_feat = keep_weekdays_only(ext_feat, date_col="Date")

            # ETF 거래일 기준으로 붙임
            base_df = base_df.merge(ext_feat, on="Date", how="left")

        except Exception as e:
            print(f"[SKIP] {name} / {ticker} 로드 실패:", e)

    # =========================================================
    # 4. 날짜 정렬 + 주말 최종 제거
    # =========================================================
    base_df = keep_weekdays_only(base_df, date_col="Date")

    # =========================================================
    # 4.1. 외부 지표 NA 보정
    # - ETF 본체는 건드리지 않고
    # - 외부 지표만 ffill
    # =========================================================
    etf_prefix = f"{etf_code}_"

    external_cols = [
        col for col in base_df.columns
        if col != "Date" and not col.startswith(etf_prefix)
    ]

    base_df[external_cols] = base_df[external_cols].ffill()

    # =========================================================
    # 5. feature_cols 정리
    # =========================================================
    close_col = f"{etf_code}_adj_close"

    feature_cols = [
        col for col in base_df.columns
        if col not in ["Date", close_col]
    ]

    return base_df, feature_cols, close_col

################################################################################################################################################
## 2. VIF 기반 불필요 칼럼 제거
################################################################################################################################################
def reduce_features_by_vif(
    df,
    feature_cols,
    vif_threshold=30.0,
    date_col="Date",
    verbose=True
):
    """
    VIF 기준으로 다중공선성이 높은 feature를 반복 제거한다.

    주의:
    - target 생성 전 단계에서 실행한다.
    - lag 생성 전 단계에서 실행한다.
    - Date, adj_close 등 보존 컬럼은 feature_cols에 넣지 않는 것을 전제로 한다.
    """

    # 1. 숫자형 feature만 사용
    numeric_feature_cols = [
        col for col in feature_cols
        if col in df.columns and pd.api.types.is_numeric_dtype(df[col])
    ]

    work_df = df[numeric_feature_cols].copy()

    # 2. inf 처리
    work_df = work_df.replace([np.inf, -np.inf], np.nan)

    # 3. VIF 계산용 결측 제거
    #    여기서는 VIF 계산에만 dropna를 쓰고,
    #    원본 df 자체를 줄이지는 않는다.
    vif_calc_df = work_df.dropna(axis=0).copy()

    print("VIF 계산 대상 row 수:", len(vif_calc_df))
    print("VIF 계산 대상 feature 수:", len(numeric_feature_cols))

    if len(vif_calc_df) == 0:
        raise ValueError("VIF 계산 가능한 데이터가 없습니다. 결측값을 확인하세요.")

    # 4. 상수 컬럼 제거
    nunique = vif_calc_df.nunique()
    constant_cols = nunique[nunique <= 1].index.tolist()

    if len(constant_cols) > 0:
        print("상수 컬럼 제거:", constant_cols)

    remaining_cols = [
        col for col in numeric_feature_cols
        if col not in constant_cols
    ]

    removed_records = []

    # 5. VIF 반복 제거
    while True:
        if len(remaining_cols) <= 1:
            break

        X = vif_calc_df[remaining_cols].copy()

        # 표준화
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        vif_values = []

        for i, col in enumerate(remaining_cols):
            try:
                vif = variance_inflation_factor(X_scaled, i)
            except Exception:
                vif = np.inf

            vif_values.append({
                "feature": col,
                "vif": vif
            })

        vif_df = pd.DataFrame(vif_values).sort_values("vif", ascending=False)

        max_vif_row = vif_df.iloc[0]
        max_feature = max_vif_row["feature"]
        max_vif = max_vif_row["vif"]

        if verbose:
            print(f"현재 max VIF: {max_vif:.2f} / feature: {max_feature}")

        if max_vif <= vif_threshold:
            break

        # 가장 VIF 높은 컬럼 제거
        remaining_cols.remove(max_feature)

        removed_records.append({
            "removed_feature": max_feature,
            "vif": max_vif,
            "remaining_feature_count": len(remaining_cols)
        })

    removed_vif_df = pd.DataFrame(removed_records)

    # 6. 최종 VIF 테이블 계산
    final_vif_records = []

    if len(remaining_cols) > 1:
        X_final = vif_calc_df[remaining_cols].copy()

        scaler = StandardScaler()
        X_final_scaled = scaler.fit_transform(X_final)

        for i, col in enumerate(remaining_cols):
            try:
                vif = variance_inflation_factor(X_final_scaled, i)
            except Exception:
                vif = np.inf

            final_vif_records.append({
                "feature": col,
                "vif": vif
            })

        final_vif_df = pd.DataFrame(final_vif_records).sort_values("vif", ascending=False)

    else:
        final_vif_df = pd.DataFrame({
            "feature": remaining_cols,
            "vif": [np.nan] * len(remaining_cols)
        })

    print()
    print("========== VIF 제거 결과 ==========")
    print("초기 feature 수:", len(numeric_feature_cols))
    print("상수 제거 feature 수:", len(constant_cols))
    print("VIF 제거 feature 수:", len(removed_records))
    print("최종 feature 수:", len(remaining_cols))
    print("===================================")

    return remaining_cols, removed_vif_df, final_vif_df


################################################################################################################################################
## 3. Target 변수 생성
################################################################################################################################################
def add_target_column(
    df,
    close_col,
    n_days=5,
    threshold=0.05,
    target_col=None
):
    """
    현재 시점 기준 n_days 뒤 수익률이 threshold 이상이면 1, 아니면 0인 target 생성.

    예:
    n_days=5, threshold=0.05
    → 5거래일 뒤 수익률이 +5% 이상이면 target=1
    """

    result = df.copy()
    result = result.sort_values("Date").reset_index(drop=True)

    if target_col is None:
        target_col = f"target_{n_days}d_up_{int(threshold * 100)}pct"

    # 미래 가격
    future_close = result[close_col].shift(-n_days)

    # 미래 수익률
    result[f"future_ret_{n_days}d"] = future_close / result[close_col] - 1

    # target 생성
    result[target_col] = np.where(
        result[f"future_ret_{n_days}d"] >= threshold,
        1,
        0
    )

    # 마지막 n_days개는 미래 가격이 없으므로 제거
    result.loc[result[f"future_ret_{n_days}d"].isna(), target_col] = np.nan

    return result, target_col


################################################################################################################################################
## 4. Lag 생성 후 변수별 최적 lag 탐색
################################################################################################################################################

def find_best_lag_by_feature(
    df,
    feature_cols,
    target_col,
    lag_days=[1, 3, 5, 10, 20],
    date_col="Date",
    method="corr"
):
    """
    각 feature별로 target과 가장 관계가 강한 선행 lag를 찾는다.

    lag 의미:
    - lag=1  : feature의 1거래일 전 값으로 오늘 target 설명
    - lag=5  : feature의 5거래일 전 값으로 오늘 target 설명
    - lag=20 : feature의 20거래일 전 값으로 오늘 target 설명

    즉, feature가 먼저 움직이고 나중에 target이 움직이는 구조만 본다.
    """

    records = []

    for col in feature_cols:
        if col not in df.columns:
            continue

        for lag in lag_days:
            temp = df[[date_col, col, target_col]].copy()

            # 선행변수 구조
            temp[f"{col}_lag{lag}"] = temp[col].shift(lag)

            temp = temp[[f"{col}_lag{lag}", target_col]].replace(
                [np.inf, -np.inf],
                np.nan
            ).dropna()

            if len(temp) < 30:
                continue

            x = temp[f"{col}_lag{lag}"]
            y = temp[target_col]

            if x.nunique() <= 1:
                corr = np.nan
            else:
                corr = x.corr(y)

            records.append({
                "feature": col,
                "lag": lag,
                "corr": corr,
                "abs_corr": abs(corr) if pd.notna(corr) else np.nan,
                "n_rows": len(temp)
            })

    lag_result_df = pd.DataFrame(records)

    if lag_result_df.empty:
        raise ValueError("lag 탐색 결과가 비어 있습니다. feature_cols 또는 target_col을 확인하세요.")

    # feature별 abs_corr가 가장 큰 lag 선택
    best_lag_df = (
        lag_result_df
        .sort_values(["feature", "abs_corr"], ascending=[True, False])
        .groupby("feature", as_index=False)
        .head(1)
        .reset_index(drop=True)
    )

    best_lag_df = best_lag_df.sort_values("abs_corr", ascending=False).reset_index(drop=True)

    return lag_result_df, best_lag_df

def make_lagged_dataset_by_best_lag(
    df,
    best_lag_df,
    target_col,
    close_col,
    n_days=5,
    date_col="Date"
):
    """
    best_lag_df 기준으로 feature별 최적 lag를 적용한 최종 모델용 데이터셋 생성.
    """

    result = pd.DataFrame()
    result[date_col] = df[date_col]
    result[close_col] = df[close_col]

    # 확인용 미래수익률 보존
    future_ret_col = f"future_ret_{n_days}d"
    if future_ret_col in df.columns:
        result[future_ret_col] = df[future_ret_col]

    # target 보존
    result[target_col] = df[target_col]

    lagged_feature_cols = []

    for _, row in best_lag_df.iterrows():
        feature = row["feature"]
        lag = int(row["lag"])

        if feature not in df.columns:
            continue

        lagged_col = f"{feature}_lag{lag}"
        result[lagged_col] = df[feature].shift(lag)
        lagged_feature_cols.append(lagged_col)

    # 결측/무한값 제거
    result = result.replace([np.inf, -np.inf], np.nan)
    result = result.dropna().reset_index(drop=True)

    return result, lagged_feature_cols



################################################################################################################################################
## 5. RandomForest in-sample 학습 + permutation importance
################################################################################################################################################

def run_rf_permutation_importance_in_sample(
    lagged_df,
    feature_cols,
    target_col,
    date_col="Date",
    close_col=None,
    n_rf_runs=3,
    n_repeats=10,
    random_state=42
):
    """
    lagged_df 기준으로 RandomForestClassifier를 in-sample 학습한 뒤
    permutation importance를 반복 계산한다.

    핵심:
    - RF를 n_rf_runs번 학습
    - 각 RF마다 permutation을 n_repeats번 수행
    - feature별 importance raw 값을 전부 저장
    - 최종 importance_df는 총 n_rf_runs * n_repeats개의 raw importance 기준으로 계산

    예:
    n_rf_runs=3, n_repeats=10이면
    feature별 importance 값 30개를 기반으로 평균/표준편차/스코어 계산
    """

    df = lagged_df.copy()

    # =====================================================
    # 1. 모델 input / target 분리
    # =====================================================

    X = df[feature_cols].copy()
    y = df[target_col].copy()

    X = X.replace([np.inf, -np.inf], np.nan)

    model_df = pd.concat([X, y], axis=1).dropna().copy()

    X = model_df[feature_cols].copy()
    y = model_df[target_col].astype(int).copy()


    # =====================================================
    # 2. 여러 RF run + permutation raw importance 저장
    # =====================================================

    importance_records = []
    baseline_records = []

    for run in range(n_rf_runs):

        rf = RandomForestClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            max_features="sqrt",
            class_weight="balanced",
            random_state=random_state + run,
            n_jobs=-1
        )

        rf.fit(X, y)

        # =================================================
        # 3. in-sample 예측 성능 확인
        # =================================================

        pred = rf.predict(X)
        pred_proba = rf.predict_proba(X)[:, 1]

        acc = accuracy_score(y, pred)
        precision = precision_score(y, pred, zero_division=0)
        recall = recall_score(y, pred, zero_division=0)
        f1 = f1_score(y, pred, zero_division=0)
        loss = log_loss(y, pred_proba)

        cm = confusion_matrix(y, pred)

        baseline_records.append({
            "run": run + 1,
            "accuracy": acc,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "log_loss": loss,
            "pred_1_count": int((pred == 1).sum()),
            "actual_1_count": int((y == 1).sum())
        })

        # =================================================
        # 4. permutation importance
        # =================================================

        perm = permutation_importance(
            rf,
            X,
            y,
            scoring="neg_log_loss",
            n_repeats=n_repeats,
            random_state=random_state + run,
            n_jobs=-1
        )

        # 핵심:
        # perm.importances shape = (n_features, n_repeats)
        # 여기서 반복별 raw importance를 전부 저장한다.
        for i, col in enumerate(feature_cols):
            for repeat_idx, importance_value in enumerate(perm.importances[i]):
                importance_records.append({
                    "run": run + 1,
                    "repeat": repeat_idx + 1,
                    "feature": col,
                    "importance": importance_value
                })

    # =====================================================
    # 5. 결과 정리
    # =====================================================

    raw_importance_df = pd.DataFrame(importance_records)
    baseline_df = pd.DataFrame(baseline_records)

    importance_df = (
        raw_importance_df
        .groupby("feature", as_index=False)
        .agg(
            importance_mean=("importance", "mean"),
            importance_std=("importance", "std"),
            importance_var=("importance", "var"),
            importance_min=("importance", "min"),
            importance_max=("importance", "max"),
            run_count=("run", "nunique"),
            repeat_count=("repeat", "count")
        )
    )

    # 안정성 점수
    # 평균 중요도는 높고, 30회 전체 기준 표준편차는 낮을수록 높게
    importance_df["importance_score"] = (
        importance_df["importance_mean"]
        - importance_df["importance_std"].fillna(0)
    )

    importance_df = importance_df.sort_values(
        "importance_score",
        ascending=False
    ).reset_index(drop=True)

    return importance_df, raw_importance_df, baseline_df

def split_lagged_feature_name(feature_name):
    """
    feature_lag20 형태의 컬럼명을 원본 feature와 lag로 분리한다.
    """

    if "_lag" not in feature_name:
        return feature_name, np.nan

    base_name = feature_name.rsplit("_lag", 1)[0]
    lag = feature_name.rsplit("_lag", 1)[1]

    try:
        lag = int(lag)
    except:
        lag = np.nan

    return base_name, lag

## 1. 설정값

여기서 중요한 점:

- `BASE_DATE`만 바꾸면 기간이 자동으로 밀립니다.
- `THRESHOLD = 0.05` 같은 고정 target threshold는 없습니다.
- target 기준 수익률은 `TARGET_RETURN_THRESHOLD_CANDIDATES`에서 Optuna가 고릅니다.

In [27]:
# =========================================================
# 기본 설정
# =========================================================
ETF_CODE = "SMH"
START_DATE = "2017-01-01"
END_DATE = None

# 기준일: 이 날짜까지만 과거 데이터로 본다.
# 실제로 돌릴 때는 최신 데이터 마지막 날짜 또는 원하는 기준일로 바꾸면 됨.
BASE_DATE = "2026-05-31"

# 기간 분리 설정
OPTUNA_VALID_MONTHS = 6      # Optuna가 best option을 고르는 검증 기간
SIM_TEST_MONTHS = 11         # Optuna가 보지 않는 simulation test 기간

# 예측 타깃 설정
# 고정값 아님. Optuna가 여기서 n_days를 고른다.
# 예: 5면 5거래일 뒤 수익률, 20이면 20거래일 뒤 수익률
N_DAYS_CANDIDATES = [5, 10, 20]

# 고정값 아님. Optuna가 여기서 target_return_threshold를 고른다.
# 예: 0.03이면 5거래일 뒤 +3% 이상일 때 target=1
TARGET_RETURN_THRESHOLD_CANDIDATES = [0.02, 0.03, 0.04, 0.05, 0.06, 0.07]

# Feature selection 설정
VIF_THRESHOLD = 30.0
FEATURE_SELECT_YEARS = 1     # Train 종료일 기준 최근 N년으로 best lag/top feature 탐색
LAG_DAYS = [1, 3, 5, 10, 20, 40, 60, 120]
TOP_N_MAX = 40

# Importance 설정
RANDOM_STATE = 42
N_RF_RUNS = 10
N_REPEATS = 10

# Optuna 설정
N_TRIALS = 300
OBJECTIVE_METRIC = "precision_lift_recall"  # precision_lift_recall / precision_lift / f1_lift / return_score
MIN_VALID_EVAL_COUNT = 30
MIN_VALID_PRED_1_COUNT = 3

# Simulation 설정
INITIAL_CASH = 1_000_000
BUY_RATIO = 0.05
MIN_CASH_RATIO = 0.30

# 외부 지표 설정
EXTERNAL_TICKERS = {
    "QQQ": "QQQ",
    "SPY": "SPY",
    "SOXX": "SOXX",
    "NVDA": "NVDA",
    "TSM": "TSM",
    "VIX": "^VIX",
    "TNX": "^TNX",
    "DXY": "DXY",
    "GOLD": "GC=F",
    "OIL": "CL=F",
}

EXTERNAL_FEATURE_TYPES = {
    "QQQ": "price",
    "SPY": "price",
    "SOXX": "price",
    "NVDA": "price",
    "TSM": "price",
    "VIX": "risk",
    "TNX": "rate",
    "DXY": "price",
    "GOLD": "price",
    "OIL": "price",
}

# =========================================================
# 캐시 설정
# =========================================================
# True: pickle 로드 후 9번(Simulation)부터 실행
# False: 처음부터 full run
LOAD_FROM_CACHE = False

# 캐시 저장 디렉토리 (ETF + BASE_DATE 기준 자동 생성)
import pickle
from pathlib import Path as _Path

_cache_tag = f"{ETF_CODE}_{BASE_DATE.replace('-', '')}"
CACHE_DIR = _Path(f"cache_{_cache_tag}")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def save_cache(name, obj):
    path = CACHE_DIR / f"{name}.pkl"
    with open(path, "wb") as f:
        pickle.dump(obj, f)
    print(f"[CACHE SAVED] {path}")

def load_cache(name):
    path = CACHE_DIR / f"{name}.pkl"
    with open(path, "rb") as f:
        obj = pickle.load(f)
    print(f"[CACHE LOADED] {path}")
    return obj


## 2. BASE_DATE 기준 동적 기간 분리

하드코딩 금지.  
`BASE_DATE`, `OPTUNA_VALID_MONTHS`, `SIM_TEST_MONTHS`만으로 기간을 계산합니다.

In [28]:
def make_oos_windows(
    base_date,
    start_date="2020-01-01",
    optuna_valid_months=6,
    sim_test_months=11,
):
    '''
    BASE_DATE 기준으로 Train / Optuna Validation / Simulation Test 기간을 자동 생성한다.

    예:
    base_date=2026-05-31, optuna_valid_months=6, sim_test_months=11

    - Simulation Test   : 2025-07-01 ~ 2026-05-31
    - Optuna Validation : 2025-01-01 ~ 2025-06-30
    - Train             : 2020-01-01 ~ 2024-12-31
    '''
    base_date = pd.to_datetime(base_date)
    start_date = pd.to_datetime(start_date)

    base_period = base_date.to_period("M")

    sim_start_period = base_period - (int(sim_test_months) - 1)
    sim_start_date = sim_start_period.to_timestamp(how="start")
    sim_end_date = base_date

    valid_end_period = sim_start_period - 1
    valid_start_period = valid_end_period - (int(optuna_valid_months) - 1)

    valid_start_date = valid_start_period.to_timestamp(how="start")
    valid_end_date = valid_end_period.to_timestamp(how="end").normalize()

    train_start_date = start_date
    train_end_date = valid_start_date - pd.Timedelta(days=1)

    if train_end_date < train_start_date:
        raise ValueError("Train 기간이 비었습니다. BASE_DATE 또는 month 설정을 확인하세요.")

    return {
        "base_date": base_date,
        "train_start": train_start_date,
        "train_end": train_end_date,
        "optuna_valid_start": valid_start_date,
        "optuna_valid_end": valid_end_date,
        "sim_test_start": sim_start_date,
        "sim_test_end": sim_end_date,
    }


WINDOWS = make_oos_windows(
    base_date=BASE_DATE,
    start_date=START_DATE,
    optuna_valid_months=OPTUNA_VALID_MONTHS,
    sim_test_months=SIM_TEST_MONTHS,
)

for k, v in WINDOWS.items():
    print(f"{k}: {v.date() if hasattr(v, 'date') else v}")

# =========================================================
# 캐시 로드 (LOAD_FROM_CACHE=True인 경우)
# =========================================================
if LOAD_FROM_CACHE:
    base_df         = load_cache("base_df")
    raw_feature_cols = load_cache("raw_feature_cols")
    close_col       = load_cache("close_col")
    vif_feature_cols = load_cache("vif_feature_cols")
    feature_cache   = load_cache("feature_cache")
    optuna_result   = load_cache("optuna_result")
    best_cfg        = optuna_result["best_config"]
    best_artifacts  = feature_cache[
        (int(best_cfg["n_days"]), float(best_cfg["target_return_threshold"]))
    ]
    N_DAYS_BEST     = int(best_cfg["n_days"])
    target_col      = best_artifacts["target_col"]
    all_lagged_eval_df  = best_artifacts["all_lagged_eval_df"]
    all_lagged_infer_df = best_artifacts["all_lagged_infer_df"]
    best_feature_cols   = best_artifacts["top_feature_cols_max"][:best_cfg["top_n"]]
    print("캐시 로드 완료. 9번(Simulation Test)부터 실행하세요.")


base_date: 2026-05-31
train_start: 2017-01-01
train_end: 2024-12-31
optuna_valid_start: 2025-01-01
optuna_valid_end: 2025-06-30
sim_test_start: 2025-07-01
sim_test_end: 2026-05-31


## 3. 보조 함수

기존 함수 일부는 inference용으로 `target`이 없는 최신 row를 만들 때 불편해서,  
dropna 방식을 조절할 수 있는 lag 생성 함수를 새로 둡니다.

In [29]:
def make_lagged_dataset_by_best_lag_v2(
    df,
    best_lag_df,
    target_col=None,
    close_col=None,
    n_days=5,
    date_col="Date",
    drop_target_na=True,
):
    '''
    best_lag_df 기준으로 feature별 최적 lag를 적용한다.

    - 평가용: target_col을 넣고 drop_target_na=True
    - inference용: target_col=None 또는 drop_target_na=False
    '''
    result = pd.DataFrame()
    result[date_col] = df[date_col]

    if close_col is not None and close_col in df.columns:
        result[close_col] = df[close_col]

    future_ret_col = f"future_ret_{n_days}d"
    if future_ret_col in df.columns:
        result[future_ret_col] = df[future_ret_col]

    if target_col is not None and target_col in df.columns:
        result[target_col] = df[target_col]

    lagged_feature_cols = []

    for _, row in best_lag_df.iterrows():
        feature = row["feature"]
        lag = int(row["lag"])

        if feature not in df.columns:
            continue

        lagged_col = f"{feature}_lag{lag}"
        result[lagged_col] = df[feature].shift(lag)
        lagged_feature_cols.append(lagged_col)

    result = result.replace([np.inf, -np.inf], np.nan)

    if drop_target_na and target_col is not None and target_col in result.columns:
        required_cols = [target_col] + lagged_feature_cols
        if close_col is not None and close_col in result.columns:
            required_cols.append(close_col)
        result = result.dropna(subset=required_cols).reset_index(drop=True)
    else:
        # inference에서는 feature만 있으면 되므로 target/future_ret NaN 때문에 최신 row를 버리지 않는다.
        result = result.reset_index(drop=True)

    return result, lagged_feature_cols


def safe_binary_metrics(y_true, pred, pred_proba=None):
    y_true = pd.Series(y_true).astype(int)
    pred = pd.Series(pred).astype(int)

    out = {
        "eval_count": int(len(y_true)),
        "actual_1_count": int((y_true == 1).sum()),
        "pred_1_count": int((pred == 1).sum()),
        "accuracy": accuracy_score(y_true, pred) if len(y_true) else np.nan,
        "precision": precision_score(y_true, pred, zero_division=0) if len(y_true) else np.nan,
        "recall": recall_score(y_true, pred, zero_division=0) if len(y_true) else np.nan,
        "f1": f1_score(y_true, pred, zero_division=0) if len(y_true) else np.nan,
    }

    if pred_proba is not None and len(y_true) > 0:
        try:
            out["roc_auc"] = roc_auc_score(y_true, pred_proba) if y_true.nunique() == 2 else np.nan
        except Exception:
            out["roc_auc"] = np.nan

        try:
            out["log_loss"] = log_loss(y_true, pred_proba) if y_true.nunique() == 2 else np.nan
        except Exception:
            out["log_loss"] = np.nan

    return out


def make_classifier_model(model_name="random_forest", random_state=42, model_params=None):
    if model_params is None:
        model_params = {}

    if model_name == "random_forest":
        return RandomForestClassifier(
            n_estimators=model_params.get("n_estimators", 500),
            max_depth=model_params.get("max_depth", None),
            min_samples_split=model_params.get("min_samples_split", 2),
            min_samples_leaf=model_params.get("min_samples_leaf", 1),
            max_features=model_params.get("max_features", "sqrt"),
            class_weight=model_params.get("class_weight", "balanced"),
            random_state=random_state,
            n_jobs=-1,
        )

    if model_name == "extra_trees":
        return ExtraTreesClassifier(
            n_estimators=model_params.get("n_estimators", 500),
            max_depth=model_params.get("max_depth", None),
            min_samples_split=model_params.get("min_samples_split", 2),
            min_samples_leaf=model_params.get("min_samples_leaf", 1),
            max_features=model_params.get("max_features", "sqrt"),
            class_weight=model_params.get("class_weight", "balanced"),
            random_state=random_state,
            n_jobs=-1,
        )

    if model_name == "gradient_boosting":
        return GradientBoostingClassifier(
            n_estimators=model_params.get("n_estimators", 300),
            learning_rate=model_params.get("learning_rate", 0.05),
            max_depth=model_params.get("max_depth", 3),
            min_samples_leaf=model_params.get("min_samples_leaf", 1),
            random_state=random_state,
        )

    if model_name == "hist_gradient_boosting":
        return HistGradientBoostingClassifier(
            max_iter=model_params.get("max_iter", 300),
            learning_rate=model_params.get("learning_rate", 0.05),
            max_leaf_nodes=model_params.get("max_leaf_nodes", 31),
            min_samples_leaf=model_params.get("min_samples_leaf", 20),
            random_state=random_state,
        )

    raise ValueError(f"Unknown model_name: {model_name}")


def suggest_model_params(trial, model_name):
    if model_name in ["random_forest", "extra_trees"]:
        return {
            "n_estimators": trial.suggest_int(f"{model_name}_n_estimators", 200, 800, step=100),
            "max_depth": trial.suggest_categorical(f"{model_name}_max_depth", [None, 3, 5, 7, 10]),
            "min_samples_split": trial.suggest_int(f"{model_name}_min_samples_split", 2, 10),
            "min_samples_leaf": trial.suggest_int(f"{model_name}_min_samples_leaf", 1, 10),
            "max_features": trial.suggest_categorical(f"{model_name}_max_features", ["sqrt", "log2"]),
            "class_weight": trial.suggest_categorical(f"{model_name}_class_weight", ["balanced", "balanced_subsample", None]),
        }

    if model_name == "gradient_boosting":
        return {
            "n_estimators": trial.suggest_int("gb_n_estimators", 100, 600, step=100),
            "learning_rate": trial.suggest_float("gb_learning_rate", 0.01, 0.2, log=True),
            "max_depth": trial.suggest_int("gb_max_depth", 2, 5),
            "min_samples_leaf": trial.suggest_int("gb_min_samples_leaf", 1, 20),
        }

    if model_name == "hist_gradient_boosting":
        return {
            "max_iter": trial.suggest_int("hgb_max_iter", 100, 600, step=100),
            "learning_rate": trial.suggest_float("hgb_learning_rate", 0.01, 0.2, log=True),
            "max_leaf_nodes": trial.suggest_int("hgb_max_leaf_nodes", 15, 63),
            "min_samples_leaf": trial.suggest_int("hgb_min_samples_leaf", 10, 50),
        }

    raise ValueError(model_name)


def evaluate_model_on_period(
    train_df,
    eval_df,
    feature_cols,
    target_col,
    close_col,
    n_days,
    model_name,
    model_params,
    pred_threshold,
    random_state=42,
):
    keep_train_cols = [target_col] + feature_cols
    train_df = train_df[keep_train_cols].replace([np.inf, -np.inf], np.nan).dropna().copy()

    keep_eval_cols = ["Date", target_col, f"future_ret_{n_days}d", close_col] + feature_cols
    eval_df = eval_df[keep_eval_cols].replace([np.inf, -np.inf], np.nan).dropna().copy()

    if len(train_df) == 0 or len(eval_df) == 0:
        raise ValueError("train/eval 데이터가 비었습니다.")

    if train_df[target_col].nunique() < 2:
        raise ValueError("train target class가 1개뿐입니다.")

    X_train = train_df[feature_cols]
    y_train = train_df[target_col].astype(int)
    X_eval = eval_df[feature_cols]
    y_eval = eval_df[target_col].astype(int)

    model = make_classifier_model(
        model_name=model_name,
        random_state=random_state,
        model_params=model_params,
    )
    model.fit(X_train, y_train)

    if hasattr(model, "predict_proba"):
        pred_proba = model.predict_proba(X_eval)[:, 1]
    else:
        pred_proba = model.predict(X_eval).astype(float)

    pred = (pred_proba >= pred_threshold).astype(int)
    metrics = safe_binary_metrics(y_eval, pred, pred_proba)

    pred_df = eval_df[["Date", close_col, f"future_ret_{n_days}d", target_col]].copy()
    pred_df["pred_proba"] = pred_proba
    pred_df["pred"] = pred

    return model, metrics, pred_df


def compute_objective_score(metrics, pred_df, objective_metric="precision_lift_recall", n_days=5):
    '''
    Optuna objective score.

    핵심:
    - precision 자체가 아니라 baseline 대비 개선분을 본다.
    - baseline = validation 구간의 actual positive rate
    - precision_lift = precision - positive_rate
    - 최종 추천 score = precision_lift * recall

    의미:
    - 전부 1 예측:
      precision == positive_rate, precision_lift == 0, recall == 1
      score = 0

    - 완벽 예측:
      precision = 1, recall = 1
      score = (1 - positive_rate) * 1

    - 딱 1개만 맞춘 모델:
      precision_lift는 높을 수 있지만 recall이 낮아서 score가 낮아짐
    '''
    eval_count = int(metrics.get("eval_count", 0) or 0)
    actual_1_count = int(metrics.get("actual_1_count", 0) or 0)
    pred_1_count = int(metrics.get("pred_1_count", 0) or 0)

    if eval_count < MIN_VALID_EVAL_COUNT:
        return 0.0

    # 매수 신호가 너무 적은 후보는 실전 검증이 어려우므로 제외
    if pred_1_count < MIN_VALID_PRED_1_COUNT:
        return 0.0

    positive_rate = actual_1_count / eval_count if eval_count > 0 else 0.0
    precision = float(metrics.get("precision", 0.0) or 0.0)
    recall = float(metrics.get("recall", 0.0) or 0.0)
    f1 = float(metrics.get("f1", 0.0) or 0.0)

    precision_lift = precision - positive_rate
    f1_lift = f1 - positive_rate

    # baseline보다 못하면 0점 처리
    precision_lift_score = max(0.0, precision_lift)
    f1_lift_score = max(0.0, f1_lift)

    if objective_metric == "precision_lift_recall":
        return precision_lift_score * recall

    if objective_metric == "precision_lift":
        return precision_lift_score

    if objective_metric == "f1_lift":
        return f1_lift_score

    if objective_metric == "return_score":
        eval_df = pred_df.copy()
        strategy_ret = np.where(eval_df["pred"] == 1, eval_df[f"future_ret_{n_days}d"], 0.0)
        compound_ret = float(np.prod(1 + strategy_ret) - 1)
        # 수익률 + baseline 대비 예측 개선을 같이 반영
        return compound_ret + 0.5 * (precision_lift_score * recall)

    raise ValueError(f"Unknown objective_metric: {objective_metric}")

## 4. Base Feature Dataset 생성

여기까지는 target을 만들지 않습니다.  
target threshold는 Optuna가 고르므로, target 생성은 trial 안에서 합니다.

In [30]:
if not LOAD_FROM_CACHE:
    base_df, raw_feature_cols, close_col = make_base_feature_dataset(
        etf_code=ETF_CODE,
        external_tickers=EXTERNAL_TICKERS,
        external_feature_types=EXTERNAL_FEATURE_TYPES,
        start_date=START_DATE,
        end_date=END_DATE,
    )
    
    base_df["Date"] = pd.to_datetime(base_df["Date"])
    base_df = base_df[
        (base_df["Date"] >= WINDOWS["train_start"]) &
        (base_df["Date"] <= WINDOWS["base_date"])
    ].sort_values("Date").reset_index(drop=True)
    
    print("base_df shape:", base_df.shape)
    print("기간:", base_df["Date"].min(), "~", base_df["Date"].max())
    print("close_col:", close_col)
    print("raw feature 수:", len(raw_feature_cols))
    
    display_df(base_df, 5)

    # 캐시 저장
    save_cache("base_df", base_df)
    save_cache("raw_feature_cols", raw_feature_cols)
    save_cache("close_col", close_col)


Loading external ticker: QQQ / QQQ
Loading external ticker: SPY / SPY
Loading external ticker: SOXX / SOXX
Loading external ticker: NVDA / NVDA
Loading external ticker: TSM / TSM
Loading external ticker: VIX / ^VIX
Loading external ticker: TNX / ^TNX
Loading external ticker: DXY / DXY
[SKIP] DXY / DXY 로드 실패: 'timestamp'
Loading external ticker: GOLD / GC=F
Loading external ticker: OIL / CL=F
base_df shape: (2364, 30)
기간: 2017-01-03 00:00:00 ~ 2026-05-29 00:00:00
close_col: SMH_adj_close
raw feature 수: 28


,Date,SMH_adj_close,SMH_ret_1d,SMH_ret_5d,SMH_ret_20d,SMH_ma5_ratio,SMH_ma20_ratio,SMH_ma60_ratio,SMH_vol_20d,SMH_volume_ratio_20d,...,VIX_level,VIX_chg_5d,VIX_chg_20d,TNX_level,TNX_diff_5d,TNX_diff_20d,GOLD_ret_5d,GOLD_ret_20d,OIL_ret_5d,OIL_ret_20d
0,2017-01-03,32.949318,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,12.85,NaN,NaN,2.450,NaN,NaN,NaN,NaN,NaN,NaN
1,2017-01-04,33.054920,0.003205,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,11.85,NaN,NaN,2.452,NaN,NaN,NaN,NaN,NaN,NaN
2,2017-01-05,32.862083,-0.005834,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,11.67,NaN,NaN,2.368,NaN,NaN,NaN,NaN,NaN,NaN
3,2017-01-06,33.031971,0.005170,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,11.32,NaN,NaN,2.418,NaN,NaN,NaN,NaN,NaN,NaN
4,2017-01-09,33.413067,0.011537,NaN,NaN,0.01061,NaN,NaN,NaN,NaN,...,11.56,NaN,NaN,2.376,NaN,NaN,NaN,NaN,NaN,NaN


## 5. Train 기준 VIF 제거

VIF는 target 없이 계산할 수 있으므로 trial마다 다시 할 필요가 없습니다.  
단, **Train 구간만 사용**합니다. Optuna Validation / Simulation Test는 보지 않습니다.

In [31]:
if not LOAD_FROM_CACHE:
    train_base_df = base_df[
        (base_df["Date"] >= WINDOWS["train_start"]) &
        (base_df["Date"] <= WINDOWS["train_end"])
    ].copy()
    
    vif_feature_cols, removed_vif_df, final_vif_df = reduce_features_by_vif(
        df=train_base_df,
        feature_cols=raw_feature_cols,
        vif_threshold=VIF_THRESHOLD,
        date_col="Date",
        verbose=True,
    )
    
    print("VIF 통과 feature 수:", len(vif_feature_cols))
    display_df(final_vif_df, 20)

    # 캐시 저장
    save_cache("vif_feature_cols", vif_feature_cols)


VIF 계산 대상 row 수: 1953
VIF 계산 대상 feature 수: 28
현재 max VIF: 153.27 / feature: SMH_ret_5d
현재 max VIF: 108.85 / feature: SMH_ret_20d
현재 max VIF: 14.84 / feature: QQQ_ret_20d

========== VIF 제거 결과 ==========
초기 feature 수: 28
상수 제거 feature 수: 0
VIF 제거 feature 수: 2
최종 feature 수: 26
VIF 통과 feature 수: 26


,feature,vif
7,QQQ_ret_20d,14.844906
9,SPY_ret_20d,14.055156
6,QQQ_ret_5d,13.154172
11,SOXX_ret_20d,11.298664
8,SPY_ret_5d,11.114995
10,SOXX_ret_5d,10.944797
2,SMH_ma20_ratio,9.311875
1,SMH_ma5_ratio,5.792034
16,VIX_level,4.748191
13,NVDA_ret_20d,4.473633


## 6. Threshold별 Feature Selection 캐시 함수

Optuna trial마다 `target_return_threshold`가 바뀌면 target도 바뀌고,  
그에 따라 best lag / top feature도 바뀔 수 있습니다.

하지만 같은 threshold가 여러 trial에서 반복되므로, threshold별 결과는 캐시합니다.

In [32]:
feature_cache = {}

def memory_cleanup_message(message):
    print(f"[MEMORY CLEANUP] {message}")
    gc.collect()


def build_feature_artifacts_for_config(n_days, target_return_threshold, include_inference_df=False):
    '''
    (n_days, target_return_threshold) 조합별로:
    1. target 생성
    2. Train 최근 FEATURE_SELECT_YEARS 기준으로 best lag 탐색
    3. best lag 적용 lagged dataset 생성
    4. Train 최근 FEATURE_SELECT_YEARS 기준으로 permutation importance
    5. importance 순위 후보 반환

    왜 key가 threshold 하나가 아니라 (n_days, threshold)인가?
    - n_days가 바뀌면 target의 미래 수익률 기간 자체가 바뀐다.
    - threshold가 같아도 n_days=5와 n_days=20은 target이 완전히 다르다.
    - 따라서 best lag / top feature도 조합별로 다시 찾아야 한다.

    메모리 절약 원칙:
    - Optuna 중에는 raw_importance_df / lag_result_df / baseline_df / target_df를 캐시에 저장하지 않는다.
    - all_lagged_infer_df는 best config 확정 후 필요할 때만 생성한다.
    - 캐시에 저장하지 않는 대형 객체는 del 후 gc.collect()를 호출하고 메시지를 남긴다.
    '''
    cache_key = (int(n_days), float(target_return_threshold))

    # 이미 만들어진 n_days + threshold 결과는 재사용
    if cache_key in feature_cache:
        artifacts = feature_cache[cache_key]

        # inference용 df가 필요한데 아직 없으면 그때만 생성
        if include_inference_df and "all_lagged_infer_df" not in artifacts:
            print(f"\n[inference lagged df build] n_days={n_days}, target_return_threshold={target_return_threshold}")
            target_df, _ = add_target_column(
                df=base_df,
                close_col=close_col,
                n_days=n_days,
                threshold=target_return_threshold,
                target_col=None,
            )

            all_lagged_infer_df, _ = make_lagged_dataset_by_best_lag_v2(
                df=target_df,
                best_lag_df=artifacts["best_lag_df"],
                target_col=artifacts["target_col"],
                close_col=close_col,
                n_days=n_days,
                date_col="Date",
                drop_target_na=False,
            )

            artifacts["all_lagged_infer_df"] = all_lagged_infer_df

            del target_df
            memory_cleanup_message(
                f"target_df deleted after building all_lagged_infer_df for n_days={n_days}, threshold={target_return_threshold}"
            )

        return artifacts

    print(f"\n[feature selection] n_days={n_days}, target_return_threshold={target_return_threshold}")

    # 1) target 생성
    target_df, target_col = add_target_column(
        df=base_df,
        close_col=close_col,
        n_days=n_days,
        threshold=target_return_threshold,
        target_col=None,
    )

    # 2) feature selection 기간: Train 종료일 기준 최근 N년
    fs_end = WINDOWS["train_end"]
    fs_start = fs_end - pd.DateOffset(years=FEATURE_SELECT_YEARS)

    feature_select_df = target_df[
        (target_df["Date"] >= fs_start) &
        (target_df["Date"] <= fs_end)
    ].copy()

    print("feature selection 기간:", feature_select_df["Date"].min(), "~", feature_select_df["Date"].max())
    print("feature selection shape:", feature_select_df.shape)

    # target 분포 확인
    print("target 분포:")
    print(feature_select_df[target_col].value_counts(dropna=False))

    # 3) best lag 탐색
    lag_result_df, best_lag_df = find_best_lag_by_feature(
        df=feature_select_df,
        feature_cols=vif_feature_cols,
        target_col=target_col,
        lag_days=LAG_DAYS,
        date_col="Date",
    )

    # feature_select_df는 lag 탐색 이후 더 이상 필요 없음
    del feature_select_df
    memory_cleanup_message(
        f"feature_select_df deleted after best lag search for n_days={n_days}, threshold={target_return_threshold}"
    )

    # 4) best lag를 전체 target_df에 적용: 평가용만 생성
    all_lagged_eval_df, lagged_feature_cols = make_lagged_dataset_by_best_lag_v2(
        df=target_df,
        best_lag_df=best_lag_df,
        target_col=target_col,
        close_col=close_col,
        n_days=n_days,
        date_col="Date",
        drop_target_na=True,
    )

    # 5) importance는 Train 최근 N년 lagged 데이터로만 계산
    feature_select_lagged_df = all_lagged_eval_df[
        (all_lagged_eval_df["Date"] >= fs_start) &
        (all_lagged_eval_df["Date"] <= fs_end)
    ].copy()

    if feature_select_lagged_df[target_col].nunique() < 2:
        raise ValueError(f"n_days={n_days}, threshold={target_return_threshold}에서 feature selection target class가 1개뿐입니다.")

    importance_df, raw_importance_df, baseline_df = run_rf_permutation_importance_in_sample(
        lagged_df=feature_select_lagged_df,
        feature_cols=lagged_feature_cols,
        target_col=target_col,
        date_col="Date",
        close_col=close_col,
        n_rf_runs=N_RF_RUNS,
        n_repeats=N_REPEATS,
        random_state=RANDOM_STATE,
    )

    # 원본 feature/lag 정보 붙이기
    top_feature_df = importance_df.copy()
    top_feature_df[["base_feature", "selected_lag"]] = top_feature_df["feature"].apply(
        lambda x: pd.Series(split_lagged_feature_name(x))
    )

    best_lag_for_merge = best_lag_df.rename(columns={
        "feature": "base_feature",
        "lag": "best_lag",
        "corr": "lag_corr",
        "abs_corr": "lag_abs_corr",
        "n_rows": "lag_n_rows",
    })

    top_feature_df = top_feature_df.merge(
        best_lag_for_merge,
        on="base_feature",
        how="left",
    )

    top_feature_cols_max = top_feature_df["feature"].head(TOP_N_MAX).tolist()

    # Optuna 중 필요한 최소 객체만 캐시
    artifacts = {
        "n_days": int(n_days),
        "target_return_threshold": target_return_threshold,
        "target_col": target_col,
        "best_lag_df": best_lag_df,
        "all_lagged_eval_df": all_lagged_eval_df,
        "lagged_feature_cols": lagged_feature_cols,
        "importance_df": importance_df,
        "top_feature_df": top_feature_df,
        "top_feature_cols_max": top_feature_cols_max,
        "feature_select_start": fs_start,
        "feature_select_end": fs_end,
    }

    # include_inference_df=True일 때만 inference용 lagged df 생성
    if include_inference_df:
        all_lagged_infer_df, _ = make_lagged_dataset_by_best_lag_v2(
            df=target_df,
            best_lag_df=best_lag_df,
            target_col=target_col,
            close_col=close_col,
            n_days=n_days,
            date_col="Date",
            drop_target_na=False,
        )
        artifacts["all_lagged_infer_df"] = all_lagged_infer_df

    feature_cache[cache_key] = artifacts

    # 캐시하지 않는 대형 객체 삭제
    del target_df
    del lag_result_df
    del raw_importance_df
    del baseline_df
    del feature_select_lagged_df
    del best_lag_for_merge

    memory_cleanup_message(
        f"non-cached objects deleted after feature artifacts build for n_days={n_days}, threshold={target_return_threshold}"
    )

    return artifacts


# 이전 함수명으로 호출하는 실수를 막기 위한 alias.
# 기존 코드 호환이 필요하면 아래 함수도 사용할 수 있지만, 내부 key는 (n_days, threshold)를 쓴다.
def build_feature_artifacts_for_threshold(target_return_threshold, include_inference_df=False):
    raise RuntimeError(
        "N_DAYS도 Optuna 대상입니다. build_feature_artifacts_for_config(n_days, target_return_threshold, ...)를 사용하세요."
    )

## 7. Optuna Validation

여기서 Optuna가 고르는 것:

- `n_days`: 몇 거래일 뒤를 볼지\n- `target_return_threshold`: target을 만드는 수익률 기준
- `model_name`
- 모델 하이퍼파라미터
- `top_n`
- `pred_proba_threshold`

Optuna가 보는 기간은 **Optuna Validation 구간까지만**입니다.  
Simulation Test 기간은 여기서 절대 사용하지 않습니다.

In [33]:
def run_optuna_validation(n_trials=30, objective_metric="precision", random_state=42):
    trial_rows = []

    def objective(trial):
        n_days = trial.suggest_categorical(
            "n_days",
            N_DAYS_CANDIDATES,
        )

        target_return_threshold = trial.suggest_categorical(
            "target_return_threshold",
            TARGET_RETURN_THRESHOLD_CANDIDATES,
        )

        try:
            artifacts = build_feature_artifacts_for_config(
                n_days=n_days,
                target_return_threshold=target_return_threshold,
            )
            target_col = artifacts["target_col"]
            all_lagged_eval_df = artifacts["all_lagged_eval_df"]
            top_feature_cols_max = artifacts["top_feature_cols_max"]

            if len(top_feature_cols_max) == 0:
                return 0.0

            max_top_n = min(TOP_N_MAX, len(top_feature_cols_max))
            top_n = trial.suggest_int("top_n", 5, max_top_n)
            feature_cols = top_feature_cols_max[:top_n]

            model_name = trial.suggest_categorical(
                "model_name",
                ["random_forest", "extra_trees", "gradient_boosting", "hist_gradient_boosting"],
            )
            model_params = suggest_model_params(trial, model_name)

            pred_threshold = trial.suggest_float("pred_proba_threshold", 0.30, 0.80, step=0.05)

            train_df = all_lagged_eval_df[
                (all_lagged_eval_df["Date"] >= WINDOWS["train_start"]) &
                (all_lagged_eval_df["Date"] <= WINDOWS["train_end"])
            ].copy()

            valid_df = all_lagged_eval_df[
                (all_lagged_eval_df["Date"] >= WINDOWS["optuna_valid_start"]) &
                (all_lagged_eval_df["Date"] <= WINDOWS["optuna_valid_end"])
            ].copy()

            _, metrics, pred_df = evaluate_model_on_period(
                train_df=train_df,
                eval_df=valid_df,
                feature_cols=feature_cols,
                target_col=target_col,
                close_col=close_col,
                n_days=n_days,
                model_name=model_name,
                model_params=model_params,
                pred_threshold=pred_threshold,
                random_state=random_state,
            )

            score = compute_objective_score(
                metrics=metrics,
                pred_df=pred_df,
                objective_metric=objective_metric,
                n_days=n_days,
            )

            positive_rate = metrics["actual_1_count"] / metrics["eval_count"] if metrics["eval_count"] > 0 else np.nan
            precision_lift = metrics["precision"] - positive_rate if pd.notna(positive_rate) else np.nan
            f1_lift = metrics["f1"] - positive_rate if pd.notna(positive_rate) else np.nan
            precision_lift_recall = max(0.0, precision_lift) * metrics["recall"] if pd.notna(precision_lift) else np.nan
            pred_1_ratio = metrics["pred_1_count"] / metrics["eval_count"] if metrics["eval_count"] > 0 else np.nan

            row = {
                "trial_number": trial.number,
                "score": score,
                "n_days": n_days,
                "target_return_threshold": target_return_threshold,
                "target_col": target_col,
                "model_name": model_name,
                "top_n": top_n,
                "pred_proba_threshold": pred_threshold,
                "positive_rate": positive_rate,
                "precision_lift": precision_lift,
                "f1_lift": f1_lift,
                "precision_lift_recall": precision_lift_recall,
                "pred_1_ratio": pred_1_ratio,
                **metrics,
            }
            row["model_params_json"] = json.dumps(model_params, ensure_ascii=False, default=str)
            trial_rows.append(row)

            trial.set_user_attr("n_days", n_days)
            trial.set_user_attr("target_col", target_col)
            trial.set_user_attr("model_params", model_params)
            trial.set_user_attr("metrics", metrics)

            return score

        except Exception as e:
            positive_rate = metrics["actual_1_count"] / metrics["eval_count"] if metrics["eval_count"] > 0 else np.nan
            precision_lift = metrics["precision"] - positive_rate if pd.notna(positive_rate) else np.nan
            f1_lift = metrics["f1"] - positive_rate if pd.notna(positive_rate) else np.nan
            precision_lift_recall = max(0.0, precision_lift) * metrics["recall"] if pd.notna(precision_lift) else np.nan
            pred_1_ratio = metrics["pred_1_count"] / metrics["eval_count"] if metrics["eval_count"] > 0 else np.nan

            row = {
                "trial_number": trial.number,
                "score": 0.0,
                "error": str(e),
                "n_days": trial.params.get("n_days", None),
                "target_return_threshold": trial.params.get("target_return_threshold", None),
            }
            trial_rows.append(row)
            return 0.0

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials)

    trials_df = pd.DataFrame(trial_rows).sort_values("score", ascending=False).reset_index(drop=True)

    best_trial = study.best_trial
    best_n_days = best_trial.params["n_days"]
    best_target_return_threshold = best_trial.params["target_return_threshold"]
    best_model_name = best_trial.params["model_name"]
    best_model_params = best_trial.user_attrs["model_params"]
    best_top_n = best_trial.params["top_n"]
    best_pred_threshold = best_trial.params["pred_proba_threshold"]

    best_config = {
        "n_days": best_n_days,
        "target_return_threshold": best_target_return_threshold,
        "target_col": best_trial.user_attrs["target_col"],
        "model_name": best_model_name,
        "model_params": best_model_params,
        "top_n": best_top_n,
        "pred_proba_threshold": best_pred_threshold,
        "score": best_trial.value,
        "objective_metric": objective_metric,
    }

    return {
        "study": study,
        "trials_df": trials_df,
        "best_config": best_config,
    }


optuna_result = run_optuna_validation(
    n_trials=N_TRIALS,
    objective_metric=OBJECTIVE_METRIC,
    random_state=RANDOM_STATE,
)

print("Best config:")
print(json.dumps(optuna_result["best_config"], ensure_ascii=False, indent=2, default=str))

display_df(optuna_result["trials_df"], 20)

if not LOAD_FROM_CACHE:
    save_cache("optuna_result", optuna_result)
    save_cache("feature_cache", feature_cache)


[I 2026-06-03 20:40:56,374] A new study created in memory with name: no-name-a46ab275-e48e-4766-ad95-d70efc4d9e1e



[feature selection] n_days=20, target_return_threshold=0.05
feature selection 기간: 2024-01-02 00:00:00 ~ 2024-12-31 00:00:00
feature selection shape: (252, 32)
target 분포:
target_20d_up_5pct
0.0    150
1.0    102
Name: count, dtype: int64
[MEMORY CLEANUP] feature_select_df deleted after best lag search for n_days=20, threshold=0.05
[MEMORY CLEANUP] non-cached objects deleted after feature artifacts build for n_days=20, threshold=0.05


[I 2026-06-03 20:41:31,075] Trial 0 finished with value: 0.0 and parameters: {'n_days': 20, 'target_return_threshold': 0.05, 'top_n': 20, 'model_name': 'random_forest', 'random_forest_n_estimators': 400, 'random_forest_max_depth': 5, 'random_forest_min_samples_split': 10, 'random_forest_min_samples_leaf': 3, 'random_forest_max_features': 'log2', 'random_forest_class_weight': None, 'pred_proba_threshold': 0.7}. Best is trial 0 with value: 0.0.



[feature selection] n_days=20, target_return_threshold=0.02
feature selection 기간: 2024-01-02 00:00:00 ~ 2024-12-31 00:00:00
feature selection shape: (252, 32)
target 분포:
target_20d_up_2pct
1.0    134
0.0    118
Name: count, dtype: int64
[MEMORY CLEANUP] feature_select_df deleted after best lag search for n_days=20, threshold=0.02
[MEMORY CLEANUP] non-cached objects deleted after feature artifacts build for n_days=20, threshold=0.02


[I 2026-06-03 20:42:05,903] Trial 1 finished with value: 0.0 and parameters: {'n_days': 20, 'target_return_threshold': 0.02, 'top_n': 13, 'model_name': 'random_forest', 'random_forest_n_estimators': 800, 'random_forest_max_depth': 7, 'random_forest_min_samples_split': 2, 'random_forest_min_samples_leaf': 1, 'random_forest_max_features': 'sqrt', 'random_forest_class_weight': None, 'pred_proba_threshold': 0.8}. Best is trial 0 with value: 0.0.


...

[I 2026-06-03 20:54:31,369] Trial 47 finished with value: 0.0 and parameters: {'n_days': 20, 'target_return_threshold': 0.03, 'top_n': 17, 'model_name': 'gradient_boosting', 'gb_n_estimators': 500, 'gb_learning_rate': 0.15256880387647942, 'gb_max_depth': 3, 'gb_min_samples_leaf': 4, 'pred_proba_threshold': 0.35}. Best is trial 43 with value: 0.13337700747557663.



[feature selection] n_days=10, target_return_threshold=0.06
feature selection 기간: 2024-01-02 00:00:00 ~ 2024-12-31 00:00:00
feature selection shape: (252, 32)
target 분포:
target_10d_up_6pct
0.0    191
1.0     61
Name: count, dtype: int64
[MEMORY CLEANUP] feature_select_df deleted after best lag search for n_days=10, threshold=0.06
[MEMORY CLEANUP] non-cached objects deleted after feature artifacts build for n_days=10, threshold=0.06


[I 2026-06-03 20:55:46,439] Trial 48 finished with value: 0.0 and parameters: {'n_days': 10, 'target_return_threshold': 0.06, 'top_n': 13, 'model_name': 'gradient_boosting', 'gb_n_estimators': 300, 'gb_learning_rate': 0.06296940061247938, 'gb_max_depth': 4, 'gb_min_samples_leaf': 3, 'pred_proba_threshold': 0.4}. Best is trial 43 with value: 0.13337700747557663.
[I 2026-06-03 20:55:50,186] Trial 49 finished with value: 0.08064809592256329 and parameters: {'n_days': 20, 'target_return_threshold': 0.06, 'top_n': 12, 'model_name': 'hist_gradient_boosting', 'hgb_max_iter': 100, 'hgb_learning_rate': 0.012117850035330727, 'hgb_max_leaf_nodes': 59, 'hgb_min_samples_leaf': 10, 'pred_proba_threshold': 0.35}. Best is trial 43 with value: 0.13337700747557663.
[I 2026-06-03 20:55:54,417] Trial 50 finished with value: 0.0 and parameters: {'n_days': 20, 'target_return_threshold': 0.07, 'top_n': 12, 'model_name': 'hist_gradient_boosting', 'hgb_max_iter': 100, 'hgb_learning_rate': 0.013995684664427237,

Best config:
{
  "n_days": 20,
  "target_return_threshold": 0.06,
  "target_col": "target_20d_up_6pct",
  "model_name": "gradient_boosting",
  "model_params": {
    "n_estimators": 400,
    "learning_rate": 0.1773417358266202,
    "max_depth": 5,
    "min_samples_leaf": 8
  },
  "top_n": 14,
  "pred_proba_threshold": 0.45,
  "score": 0.2365819873855216,
  "objective_metric": "precision_lift_recall"
}


,trial_number,score,n_days,target_return_threshold,target_col,model_name,top_n,pred_proba_threshold,positive_rate,precision_lift,...,eval_count,actual_1_count,pred_1_count,accuracy,precision,recall,f1,roc_auc,log_loss,model_params_json
0,243,0.236582,20,0.06,target_20d_up_6pct,gradient_boosting,14,0.45,0.434426,0.261226,...,122,53,69,0.786885,0.695652,0.905660,0.786885,0.823899,0.817532,"{""n_estimators"": 400, ""learning_rate"": 0.17734..."
1,298,0.228988,20,0.06,target_20d_up_6pct,gradient_boosting,14,0.45,0.434426,0.282240,...,122,53,60,0.778689,0.716667,0.811321,0.761062,0.815969,0.688145,"{""n_estimators"": 400, ""learning_rate"": 0.11662..."
2,222,0.210076,20,0.06,target_20d_up_6pct,gradient_boosting,14,0.45,0.434426,0.242044,...,122,53,68,0.762295,0.676471,0.867925,0.760331,0.828548,0.676509,"{""n_estimators"": 400, ""learning_rate"": 0.11882..."
3,147,0.202060,20,0.06,target_20d_up_6pct,gradient_boosting,15,0.45,0.434426,0.223108,...,122,53,73,0.754098,0.657534,0.905660,0.761905,0.815696,0.833944,"{""n_estimators"": 400, ""learning_rate"": 0.15921..."
4,277,0.193023,20,0.06,target_20d_up_6pct,gradient_boosting,14,0.45,0.434426,0.227338,...,122,53,68,0.745902,0.661765,0.849057,0.743802,0.806946,0.883659,"{""n_estimators"": 400, ""learning_rate"": 0.18910..."
5,223,0.192803,20,0.06,target_20d_up_6pct,gradient_boosting,14,0.45,0.434426,0.232240,...,122,53,66,0.745902,0.666667,0.830189,0.739496,0.782335,0.772311,"{""n_estimators"": 400, ""learning_rate"": 0.11672..."
6,296,0.192561,20,0.06,target_20d_up_6pct,gradient_boosting,14,0.45,0.434426,0.242993,...,122,53,62,0.745902,0.677419,0.792453,0.730435,0.780421,0.720439,"{""n_estimators"": 400, ""learning_rate"": 0.11816..."
7,242,0.192561,20,0.06,target_20d_up_6pct,gradient_boosting,14,0.45,0.434426,0.242993,...,122,53,62,0.745902,0.677419,0.792453,0.730435,0.825540,0.735847,"{""n_estimators"": 400, ""learning_rate"": 0.17392..."
8,218,0.184543,20,0.06,target_20d_up_6pct,gradient_boosting,14,0.45,0.434426,0.222290,...,122,53,67,0.737705,0.656716,0.830189,0.733333,0.778234,0.813680,"{""n_estimators"": 400, ""learning_rate"": 0.11957..."
9,214,0.184261,20,0.06,target_20d_up_6pct,gradient_boosting,14,0.45,0.434426,0.227112,...,122,53,65,0.737705,0.661538,0.811321,0.728814,0.788351,0.819135,"{""n_estimators"": 400, ""learning_rate"": 0.15521..."


## 8. Best Config 기준 Feature 결과 확인

Best trial의 `n_days + target_return_threshold` 기준으로 만들어진 top feature / best lag를 확인합니다.

In [34]:
best_cfg = optuna_result["best_config"]

# best n_days + threshold가 확정된 뒤에만 inference용 lagged df까지 생성한다.
best_artifacts = build_feature_artifacts_for_config(
    n_days=best_cfg["n_days"],
    target_return_threshold=best_cfg["target_return_threshold"],
    include_inference_df=True,
)

N_DAYS_BEST = int(best_cfg["n_days"])
target_col = best_artifacts["target_col"]
all_lagged_eval_df = best_artifacts["all_lagged_eval_df"]
all_lagged_infer_df = best_artifacts["all_lagged_infer_df"]
best_feature_cols = best_artifacts["top_feature_cols_max"][:best_cfg["top_n"]]

print("best n_days:", N_DAYS_BEST)
print("best target_return_threshold:", best_cfg["target_return_threshold"])
print("target_col:", target_col)
print("best top_n:", best_cfg["top_n"])
print("사용 feature 수:", len(best_feature_cols))

display_df(best_artifacts["top_feature_df"], 30)

if not LOAD_FROM_CACHE:
    # inference_df까지 확정된 뒤 feature_cache 재저장
    save_cache("feature_cache", feature_cache)



[inference lagged df build] n_days=20, target_return_threshold=0.06
[MEMORY CLEANUP] target_df deleted after building all_lagged_infer_df for n_days=20, threshold=0.06
best n_days: 20
best target_return_threshold: 0.06
target_col: target_20d_up_6pct
best top_n: 14
사용 feature 수: 14


,feature,importance_mean,importance_std,importance_var,importance_min,importance_max,run_count,repeat_count,importance_score,base_feature,selected_lag,best_lag,lag_corr,lag_abs_corr,lag_n_rows
0,SMH_ma20_ratio_lag60,0.082162,0.007437,0.000055,0.066019,0.101213,10,100,0.074725,SMH_ma20_ratio,60,60,0.471102,0.471102,192
1,NVDA_ret_20d_lag60,0.071393,0.004668,0.000022,0.059374,0.081681,10,100,0.066725,NVDA_ret_20d,60,60,0.572862,0.572862,192
2,SMH_ma60_ratio_lag60,0.060045,0.004610,0.000021,0.047313,0.068941,10,100,0.055436,SMH_ma60_ratio,60,60,0.539456,0.539456,192
3,GOLD_ret_20d_lag40,0.060370,0.006031,0.000036,0.045321,0.075099,10,100,0.054340,GOLD_ret_20d,40,40,0.341239,0.341239,212
4,VIX_chg_5d_lag120,0.055214,0.004690,0.000022,0.045683,0.066337,10,100,0.050524,VIX_chg_5d,120,120,-0.138025,0.138025,132
5,TSM_ret_20d_lag60,0.053164,0.005361,0.000029,0.043796,0.065893,10,100,0.047803,TSM_ret_20d,60,60,0.492160,0.492160,192
6,TNX_level_lag40,0.050480,0.004365,0.000019,0.041394,0.060357,10,100,0.046116,TNX_level,40,40,0.200969,0.200969,212
7,TNX_diff_20d_lag60,0.047013,0.003716,0.000014,0.037565,0.057901,10,100,0.043297,TNX_diff_20d,60,60,0.243834,0.243834,192
8,SPY_ret_20d_lag20,0.043863,0.003115,0.000010,0.037151,0.051453,10,100,0.040747,SPY_ret_20d,20,20,-0.427429,0.427429,232
9,OIL_ret_20d_lag40,0.032070,0.002540,0.000006,0.026171,0.039321,10,100,0.029530,OIL_ret_20d,40,40,0.362590,0.362590,212


## 9. Simulation Test

Optuna에서 확정된 config를 그대로 적용합니다.  
여기서는 더 이상 threshold/model/top_n을 바꾸지 않습니다.

학습 데이터는 `Train + Optuna Validation`까지 사용하고,  
Simulation Test 기간에만 평가합니다.

In [35]:
def summarize_strategy_result(pred_df, target_col, n_days):
    eval_df = pred_df.dropna(subset=[target_col, "pred", f"future_ret_{n_days}d"]).copy()
    if len(eval_df) == 0:
        return pd.DataFrame([{"eval_count": 0}])

    metrics = safe_binary_metrics(
        y_true=eval_df[target_col].astype(int),
        pred=eval_df["pred"].astype(int),
        pred_proba=eval_df["pred_proba"],
    )

    positive_rate = metrics["actual_1_count"] / metrics["eval_count"] if metrics["eval_count"] > 0 else np.nan
    precision_lift = metrics["precision"] - positive_rate if pd.notna(positive_rate) else np.nan
    f1_lift = metrics["f1"] - positive_rate if pd.notna(positive_rate) else np.nan
    precision_lift_recall = max(0.0, precision_lift) * metrics["recall"] if pd.notna(precision_lift) else np.nan
    pred_1_ratio = metrics["pred_1_count"] / metrics["eval_count"] if metrics["eval_count"] > 0 else np.nan

    strategy_ret = np.where(eval_df["pred"] == 1, eval_df[f"future_ret_{n_days}d"], 0.0)
    strategy_ret_if = np.where(eval_df[target_col].astype(int) == 1, eval_df[f"future_ret_{n_days}d"], 0.0)

    compound_return = float(np.prod(1 + strategy_ret) - 1)
    compound_return_if = float(np.prod(1 + strategy_ret_if) - 1)

    avg_ret_when_buy = (
        float(eval_df.loc[eval_df["pred"] == 1, f"future_ret_{n_days}d"].mean())
        if (eval_df["pred"] == 1).any()
        else np.nan
    )

    avg_ret_when_actual_1 = (
        float(eval_df.loc[eval_df[target_col].astype(int) == 1, f"future_ret_{n_days}d"].mean())
        if (eval_df[target_col].astype(int) == 1).any()
        else np.nan
    )

    out = {
        "target_col_name": target_col,
        "n_days": n_days,
        **metrics,
        "positive_rate": positive_rate,
        "precision_lift": precision_lift,
        "f1_lift": f1_lift,
        "precision_lift_recall": precision_lift_recall,
        "pred_1_ratio": pred_1_ratio,
        "pred_0_count": metrics["eval_count"] - metrics["pred_1_count"],
        "avg_future_ret_all": float(eval_df[f"future_ret_{n_days}d"].mean()),
        "avg_future_ret_when_pred_1": avg_ret_when_buy,
        "strategy_compound_return_simple": compound_return,
        "actual_1_count_if": int((eval_df[target_col].astype(int) == 1).sum()),
        "avg_future_ret_when_actual_1_if": avg_ret_when_actual_1,
        "strategy_compound_return_simple_if": compound_return_if,
    }
    return pd.DataFrame([out])


def make_cash_simulation(
    pred_df,
    close_col,
    target_col,
    n_days,
    take_profit_threshold,
    initial_cash=1_000_000,
    buy_ratio=0.05,
    min_cash_ratio=0.30,
):
    '''
    익절 조건 추가된 현금 기준 시뮬레이션.

    매도 우선순위 (매일 lot별 체크):
    1. 익절: 현재가 / 매수가 - 1 >= take_profit_threshold → 즉시 매도
    2. 만기: 매수 후 n_days 거래일 경과 → 매도

    pred 기준 / actual_target_if 기준 동시 계산.
    '''
    df = pred_df.copy().sort_values("Date").reset_index(drop=True)
    min_cash = initial_cash * min_cash_ratio

    cash = float(initial_cash)
    cash_if = float(initial_cash)

    open_lots = []
    open_lots_if = []

    records = []

    for i, row in df.iterrows():
        date = row["Date"]
        price = float(row[close_col])
        actual_target = int(row[target_col]) if pd.notna(row[target_col]) else 0

        sell_amount = 0.0
        sell_amount_if = 0.0
        sell_reason = None
        sell_reason_if = None

        # -------------------------------------------------
        # 1) pred 기준: 익절 or 만기 체크
        # -------------------------------------------------
        remaining_lots = []
        for lot in open_lots:
            current_return = price / lot["buy_price"] - 1

            # 익절 조건
            if current_return >= take_profit_threshold:
                sell_value = lot["qty"] * price
                cash += sell_value
                sell_amount += sell_value
                sell_reason = "TAKE_PROFIT"

            # 만기 조건
            elif lot["sell_idx"] <= i:
                sell_value = lot["qty"] * price
                cash += sell_value
                sell_amount += sell_value
                if sell_reason != "TAKE_PROFIT":
                    sell_reason = "EXPIRE"

            else:
                remaining_lots.append(lot)

        open_lots = remaining_lots

        # -------------------------------------------------
        # 2) actual_target_if 기준: 익절 or 만기 체크
        # -------------------------------------------------
        remaining_lots_if = []
        for lot in open_lots_if:
            current_return_if = price / lot["buy_price"] - 1

            if current_return_if >= take_profit_threshold:
                sell_value_if = lot["qty"] * price
                cash_if += sell_value_if
                sell_amount_if += sell_value_if
                sell_reason_if = "TAKE_PROFIT"

            elif lot["sell_idx"] <= i:
                sell_value_if = lot["qty"] * price
                cash_if += sell_value_if
                sell_amount_if += sell_value_if
                if sell_reason_if != "TAKE_PROFIT":
                    sell_reason_if = "EXPIRE"

            else:
                remaining_lots_if.append(lot)

        open_lots_if = remaining_lots_if

        # 매도 반영 후 평가액
        holding_value = sum(lot["qty"] * price for lot in open_lots)
        asset = cash + holding_value

        holding_value_if = sum(lot["qty"] * price for lot in open_lots_if)
        asset_if = cash_if + holding_value_if

        # -------------------------------------------------
        # 3) pred 기준 매수
        # -------------------------------------------------
        action = "HOLD"
        buy_amount = 0.0
        buy_qty = 0.0
        buy_sell_due_date = pd.NaT

        if int(row["pred"]) == 1 and cash > min_cash:
            buy_amount = min(asset * buy_ratio, cash - min_cash)
            if buy_amount > 0:
                buy_qty = buy_amount / price
                cash -= buy_amount
                sell_idx = i + int(n_days)
                buy_sell_due_date = df.loc[sell_idx, "Date"] if sell_idx < len(df) else pd.NaT
                open_lots.append({
                    "buy_idx": i,
                    "sell_idx": sell_idx,
                    "buy_date": date,
                    "sell_due_date": buy_sell_due_date,
                    "qty": buy_qty,
                    "buy_price": price,
                    "buy_amount": buy_amount,
                })
                action = "BUY"

        if sell_amount > 0 and action == "BUY":
            action = f"SELL_{sell_reason}_BUY"
        elif sell_amount > 0:
            action = f"SELL_{sell_reason}"

        # -------------------------------------------------
        # 4) actual_target_if 기준 매수
        # -------------------------------------------------
        action_if = "HOLD"
        buy_amount_if = 0.0
        buy_qty_if = 0.0
        buy_sell_due_date_if = pd.NaT

        if actual_target == 1 and cash_if > min_cash:
            buy_amount_if = min(asset_if * buy_ratio, cash_if - min_cash)
            if buy_amount_if > 0:
                buy_qty_if = buy_amount_if / price
                cash_if -= buy_amount_if
                sell_idx_if = i + int(n_days)
                buy_sell_due_date_if = df.loc[sell_idx_if, "Date"] if sell_idx_if < len(df) else pd.NaT
                open_lots_if.append({
                    "buy_idx": i,
                    "sell_idx": sell_idx_if,
                    "buy_date": date,
                    "sell_due_date": buy_sell_due_date_if,
                    "qty": buy_qty_if,
                    "buy_price": price,
                    "buy_amount": buy_amount_if,
                })
                action_if = "BUY"

        if sell_amount_if > 0 and action_if == "BUY":
            action_if = f"SELL_{sell_reason_if}_BUY"
        elif sell_amount_if > 0:
            action_if = f"SELL_{sell_reason_if}"

        # 매수까지 반영 후 평가액 재계산
        holding_value = sum(lot["qty"] * price for lot in open_lots)
        asset = cash + holding_value

        holding_value_if = sum(lot["qty"] * price for lot in open_lots_if)
        asset_if = cash_if + holding_value_if

        records.append({
            "Date": date,
            "price": price,
            "pred_proba": float(row["pred_proba"]),
            "pred": int(row["pred"]),
            "actual_target": actual_target,

            # pred 기준
            "trade_action": action,
            "cash": cash,
            "buy_amount": buy_amount,
            "buy_qty": buy_qty,
            "buy_sell_due_date": buy_sell_due_date,
            "sell_amount": sell_amount,
            "holding_value": holding_value,
            "open_lot_count": len(open_lots),
            "asset": asset,
            "cum_return": asset / initial_cash - 1,

            # actual_target_if 기준
            "trade_action_if": action_if,
            "cash_if": cash_if,
            "buy_amount_if": buy_amount_if,
            "buy_qty_if": buy_qty_if,
            "buy_sell_due_date_if": buy_sell_due_date_if,
            "sell_amount_if": sell_amount_if,
            "holding_value_if": holding_value_if,
            "open_lot_count_if": len(open_lots_if),
            "asset_if": asset_if,
            "cum_return_if": asset_if / initial_cash - 1,
        })

    return pd.DataFrame(records)


def plot_pred_vs_actual(sim_pred_df, target_col, title="Prediction vs Actual Target"):
    plot_df = sim_pred_df.copy().sort_values("Date")
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=plot_df["Date"],
        y=plot_df["pred"] + 0.04,
        mode="lines+markers",
        name="pred",
        line=dict(width=2, dash="dot"),
        marker=dict(size=8, symbol="x"),
    ))

    fig.add_trace(go.Scatter(
        x=plot_df["Date"],
        y=plot_df[target_col],
        mode="lines+markers",
        name="actual_target",
        line=dict(width=2),
        marker=dict(size=8, symbol="circle"),
    ))

    fig.update_layout(
        title=title,
        xaxis_title="Date",
        yaxis_title="Value",
        hovermode="x unified",
        height=450,
    )
    fig.show()
    return fig


def plot_cash_flow(cash_sim_df, title="Cash Flow"):
    plot_df = cash_sim_df.copy().sort_values("Date")
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=plot_df["Date"],
        y=plot_df["asset"],
        mode="lines",
        name="asset_pred",
    ))

    fig.add_trace(go.Scatter(
        x=plot_df["Date"],
        y=plot_df["asset_if"],
        mode="lines",
        name="asset_actual_target_if",
    ))

    fig.update_layout(
        title=title,
        xaxis_title="Date",
        yaxis_title="Asset",
        hovermode="x unified",
        height=450,
    )
    fig.show()
    return fig


# =========================================================
# Simulation Test 실행
# =========================================================
train_plus_valid_df = all_lagged_eval_df[
    (all_lagged_eval_df["Date"] >= WINDOWS["train_start"]) &
    (all_lagged_eval_df["Date"] <= WINDOWS["optuna_valid_end"])
].copy()

sim_eval_df = all_lagged_eval_df[
    (all_lagged_eval_df["Date"] >= WINDOWS["sim_test_start"]) &
    (all_lagged_eval_df["Date"] <= WINDOWS["sim_test_end"])
].copy()

sim_model, sim_metrics, sim_pred_df = evaluate_model_on_period(
    train_df=train_plus_valid_df,
    eval_df=sim_eval_df,
    feature_cols=best_feature_cols,
    target_col=target_col,
    close_col=close_col,
    n_days=N_DAYS_BEST,
    model_name=best_cfg["model_name"],
    model_params=best_cfg["model_params"],
    pred_threshold=best_cfg["pred_proba_threshold"],
    random_state=RANDOM_STATE,
)

sim_summary_df = summarize_strategy_result(
    pred_df=sim_pred_df,
    target_col=target_col,
    n_days=N_DAYS_BEST,
)

cash_sim_df = make_cash_simulation(
    pred_df=sim_pred_df,
    close_col=close_col,
    target_col=target_col,
    n_days=N_DAYS_BEST,
    take_profit_threshold=best_cfg["target_return_threshold"],
    initial_cash=INITIAL_CASH,
    buy_ratio=BUY_RATIO,
    min_cash_ratio=MIN_CASH_RATIO,
)

print("Simulation Test 기간:", sim_eval_df["Date"].min(), "~", sim_eval_df["Date"].max())
print("target_col:", target_col)
print("n_days:", N_DAYS_BEST)
print("take_profit_threshold:", best_cfg["target_return_threshold"])
print()
print("pred 기준 action 분포:")
print(cash_sim_df["trade_action"].value_counts(dropna=False))
print()
print("actual_target_if 기준 action 분포:")
print(cash_sim_df["trade_action_if"].value_counts(dropna=False))
print()
print("Simulation metrics:")
display_df(pd.DataFrame([sim_metrics]), 5)

fig_pred_actual = plot_pred_vs_actual(
    sim_pred_df=sim_pred_df,
    target_col=target_col,
    title=f"{ETF_CODE} Simulation Test: pred / actual_target",
)

fig_cash_flow = plot_cash_flow(
    cash_sim_df=cash_sim_df,
    title=f"{ETF_CODE} Simulation Test Cash Flow: pred vs actual_target_if",
)


Simulation Test 기간: 2025-07-01 00:00:00 ~ 2026-04-30 00:00:00
target_col: target_20d_up_6pct
n_days: 20
pred 기준 매수/매도 action 분포:
trade_action
HOLD        132
BUY          37
SELL         35
SELL_BUY      6
Name: count, dtype: int64
actual_target_if 기준 매수/매도 action 분포:
trade_action_if
HOLD        96
BUY         49
SELL        33
SELL_BUY    32
Name: count, dtype: int64
Simulation metrics:


,eval_count,actual_1_count,pred_1_count,accuracy,precision,recall,f1,roc_auc,log_loss
0,210,91,43,0.514286,0.372093,0.175824,0.238806,0.499677,1.493137


## 10. Real Inference

최종 config를 사용합니다.  
실제 예측에서는 Simulation Test까지 포함한 `BASE_DATE` 이전 전체 데이터를 학습에 사용하고,  
가장 최근 row를 예측합니다.

주의: 최신 row는 미래 수익률이 없으므로 target이 NaN일 수 있습니다.  
그래서 inference용 lagged dataset은 target NaN 때문에 최신 row를 버리지 않게 만들었습니다.

In [36]:
final_train_df = all_lagged_eval_df[
    (all_lagged_eval_df["Date"] >= WINDOWS["train_start"]) &
    (all_lagged_eval_df["Date"] <= WINDOWS["base_date"])
].copy()

latest_row = all_lagged_infer_df[
    all_lagged_infer_df["Date"] <= WINDOWS["base_date"]
].sort_values("Date").tail(1).copy()

X_final = final_train_df[best_feature_cols].replace([np.inf, -np.inf], np.nan)
y_final = final_train_df[target_col].astype(int)
final_model_df = pd.concat([X_final, y_final], axis=1).dropna().copy()

if final_model_df[target_col].nunique() < 2:
    raise ValueError("final_train target class가 1개뿐입니다.")

final_model = make_classifier_model(
    model_name=best_cfg["model_name"],
    random_state=RANDOM_STATE,
    model_params=best_cfg["model_params"],
)
final_model.fit(final_model_df[best_feature_cols], final_model_df[target_col].astype(int))

X_latest = latest_row[best_feature_cols].replace([np.inf, -np.inf], np.nan)
if X_latest.isna().any(axis=None):
    nan_cols = X_latest.columns[X_latest.isna().any()].tolist()
    raise ValueError(f"latest inference feature에 NaN이 있습니다: {nan_cols}")

if hasattr(final_model, "predict_proba"):
    latest_proba = float(final_model.predict_proba(X_latest)[:, 1][0])
else:
    latest_proba = float(final_model.predict(X_latest)[0])

latest_pred = int(latest_proba >= best_cfg["pred_proba_threshold"])

real_inference_result = {
    "base_date": str(WINDOWS["base_date"].date()),
    "latest_signal_date": str(pd.to_datetime(latest_row["Date"].iloc[0]).date()),
    "etf_code": ETF_CODE,
    "n_days": N_DAYS_BEST,
    "target_return_threshold": best_cfg["target_return_threshold"],
    "target_col": target_col,
    "model_name": best_cfg["model_name"],
    "top_n": best_cfg["top_n"],
    "pred_proba_threshold": best_cfg["pred_proba_threshold"],
    "pred_proba": latest_proba,
    "pred": latest_pred,
}

real_inference_result

{'base_date': '2026-05-31',
 'latest_signal_date': '2026-05-29',
 'etf_code': 'SMH',
 'n_days': 20,
 'target_return_threshold': 0.06,
 'target_col': 'target_20d_up_6pct',
 'model_name': 'gradient_boosting',
 'top_n': 14,
 'pred_proba_threshold': 0.45,
 'pred_proba': 0.4945678974299898,
 'pred': 1}

## 11. 결과 저장

In [37]:
OUTPUT_DIR = Path("experiments_oos_dynamic_threshold_optuna")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

base_tag = pd.to_datetime(BASE_DATE).strftime("%Y%m%d")

optuna_result["trials_df"].to_excel(OUTPUT_DIR / f"1. optuna_trials_{ETF_CODE}_{base_tag}.xlsx", index=False)
best_artifacts["top_feature_df"].to_excel(OUTPUT_DIR / f"2. top_features_{ETF_CODE}_{base_tag}.xlsx", index=False)
best_artifacts["best_lag_df"].to_excel(OUTPUT_DIR / f"3. best_lag_{ETF_CODE}_{base_tag}.xlsx", index=False)
sim_pred_df.to_excel(OUTPUT_DIR / f"4. simulation_pred_{ETF_CODE}_{base_tag}.xlsx", index=False)
cash_sim_df.to_excel(OUTPUT_DIR / f"5. cash_simulation_{ETF_CODE}_{base_tag}.xlsx", index=False)
fig_pred_actual.write_html(OUTPUT_DIR / f"6. plot_pred_vs_actual_{ETF_CODE}_{base_tag}.html")
fig_cash_flow.write_html(OUTPUT_DIR / f"7. plot_cash_flow_{ETF_CODE}_{base_tag}.html")

with open(OUTPUT_DIR / f"8. best_config_{ETF_CODE}_{base_tag}.json", "w", encoding="utf-8") as f:
    json.dump(best_cfg, f, ensure_ascii=False, indent=2, default=str)

with open(OUTPUT_DIR / f"9. real_inference_{ETF_CODE}_{base_tag}.json", "w", encoding="utf-8") as f:
    json.dump(real_inference_result, f, ensure_ascii=False, indent=2, default=str)

print("저장 완료:", OUTPUT_DIR.resolve())

저장 완료: /Users/jongheelee/Desktop/JH/01. Working/1. ETF_Prediction/etf_predict/experiments_oos_dynamic_threshold_optuna
